# Figures for the bridge paper

Every panel is generated from a saved artefact (JSON / JSONL / NPZ) or from values
transcribed from a results markdown file, named in the cell that uses it.
Run: `jupyter nbconvert --to notebook --execute --inplace figures.ipynb`


In [1]:
from pathlib import Path
import json, shutil
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

ROOT = Path.cwd()
while not (ROOT / "research-phase01-ver02").exists():
    ROOT = ROOT.parent
FIG = ROOT / "draft" / "translation" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
OUT = ROOT / "draft" / "translation"

plt.rcParams.update({
    "figure.dpi": 200, "savefig.dpi": 200, "font.size": 8,
    "axes.titlesize": 8.5, "axes.labelsize": 8, "legend.fontsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.22, "grid.linewidth": 0.6,
})
BLUE, ORANGE, GREY, LGREY, RED, GREEN, PURPLE = (
    "#1f4e79", "#c55a11", "#6b6b6b", "#b8b8b8", "#b00020", "#2e7d32", "#6a3d9a")

def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG / name, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print("wrote", name)


## Assets already drawn by hand (kept byte-identical)

In [2]:
for src, dst in (("gpqa.png", "fig_benchmarks.png"), ("mapping.png", "fig_channels.png")):
    shutil.copyfile(OUT / src, FIG / dst)
    print("copied", src, "->", dst)


copied gpqa.png -> fig_benchmarks.png
copied mapping.png -> fig_channels.png


## Re-score the five channels on the first 1,000 pairs

Same scorer and same subset for every channel (BLEU 13a, chrF, reference NLL,
degeneration = unique/total ≤ 0.5). Written once to `rescored_n1000.json`; the
plotting cells below read that file.

In [3]:
import re, unicodedata
from sacrebleu.metrics import BLEU, CHRF

RESCORE = OUT / "rescored_n1000.json"
if not RESCORE.exists():
    B, C = BLEU(tokenize="13a"), CHRF()
    VI = re.compile(r"[\u00e0-\u01b0\u1ea0-\u1ef9]")
    def diag(hyp):
        deg = emp = words = 0
        for h in hyp:
            toks = h.split()
            if not toks:
                emp += 1
                continue
            if len(set(toks)) / len(toks) <= 0.5:
                deg += 1
            words += len(toks)
        return dict(empty_rate=emp / len(hyp), degenerate_rate=deg / len(hyp),
                    mean_hyp_words=words / max(len(hyp) - emp, 1))
    out, ids_ref = {}, None
    for ch in ("mapped", "lgbm", "native", "identity", "copy"):
        rows = [json.loads(l) for l in
                (ROOT / f"research-phase01-ver02/results/gen_vi_en_{ch}.jsonl").open(encoding="utf-8")][:1000]
        ids = [r["id"] for r in rows]
        ids_ref = ids if ids_ref is None else ids_ref
        assert ids == ids_ref, ch
        ref = [r["ref"] for r in rows]; hyp = [r["hyp"] for r in rows]
        nll = np.array([r.get("nll", np.nan) for r in rows], float)
        out[ch] = dict(n=len(hyp), bleu=B.corpus_score(hyp, [ref]).score,
                       chrf=C.corpus_score(hyp, [ref]).score, nll=float(np.nanmean(nll)),
                       **diag(hyp))
    RESCORE.write_text(json.dumps(out, indent=2), encoding="utf-8")
    print("wrote", RESCORE.name)
else:
    print("reusing", RESCORE.name)


That's 100 lines that end in a tokenized period ('.')


It looks like you forgot to detokenize your test data, which may hurt your score.


If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


wrote rescored_n1000.json


## Layer-wise transfer

Source: `research-phase01-ver02/results/mining.json`, keys `m4_layer_transfer_*`.

In [4]:
m = json.loads((ROOT / "research-phase01-ver02/results/mining.json").read_text(encoding="utf-8"))
fig, ax = plt.subplots(figsize=(4.6, 2.7))
for key, col, lab in (("m4_layer_transfer_vi_en", BLUE, "vi→en"),
                      ("m4_layer_transfer_en_vi", ORANGE, "en→vi")):
    ys = [m[key]["layers"][str(i)]["token_cosine_mean"] for i in range(7)]
    ax.plot(range(7), ys, marker="o", ms=4, lw=1.6, color=col, label=lab)
    ax.annotate(f"{ys[-1]:.2f}", (6, ys[-1]), textcoords="offset points",
                xytext=(4, 3), fontsize=7, color=col)
ax.set_xticks(range(7))
ax.set_xlabel("source encoder layer (0 = embeddings)")
ax.set_ylabel("token cosine to target final state")
ax.set_ylim(0, 0.55)
ax.legend(frameon=False)
ax.set_title("Only the last encoder layer carries target-compatible information")
save(fig, "fig_layers.png")


wrote fig_layers.png


## Geometry and the control that breaks cosine

Source: `research-phase01-ver02/results/geometry_vi_en.json` (5,000-sentence bank,
121,713 token pairs). The dashed line is the same value the dataset-mean control earns.

In [5]:
g = json.loads((ROOT / "research-phase01-ver02/results/geometry_vi_en.json").read_text(encoding="utf-8"))
rows = {r["variant"]: r for r in g["rows"]}
order = ["map", "lgbm", "mean", "identity"]
labels = ["ridge", "LightGBM", "mean μ", "identity"]
cols = [BLUE, ORANGE, GREY, LGREY]

fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.6))
panels = [("token_cosine", "token cosine", 0.0, 0.55),
          ("sentence_cosine", "sentence cosine", 0.0, 0.9),
          ("retrieval@1", "retrieval@1", 0.0, 0.10)]
for ax, (key, lab, lo, hi) in zip(axes, panels):
    vals = [rows[k][key] for k in order]
    ax.bar(range(4), vals, color=cols, width=0.65)
    for i, v in enumerate(vals):
        ax.annotate(f"{v:.3f}" if v < 0.1 else f"{v:.2f}", (i, v), ha="center",
                    va="bottom", fontsize=6.5, xytext=(0, 1), textcoords="offset points")
    ax.set_xticks(range(4))
    ax.set_xticklabels(labels, rotation=18, ha="right")
    ax.set_ylabel(lab)
    ax.set_ylim(lo, hi)
    if key != "retrieval@1":
        ax.axhline(vals[2], ls="--", lw=1, color=RED)
axes[0].text(1.45, 0.285, "dashed = constant $\mu_Y$", fontsize=6.5, color=RED)
axes[1].text(1.45, 0.585, "dashed = constant $\mu_Y$", fontsize=6.5, color=RED)
axes[2].axhline(g["chance_r1"], ls=":", lw=1, color=RED, label=f"chance = {g['chance_r1']:.4f}")
axes[2].legend(frameon=False, fontsize=6.2, loc="upper right")
axes[2].set_ylim(0, 0.105)
save(fig, "fig_geometry.png")


<>:23: SyntaxWarning: invalid escape sequence '\m'
<>:24: SyntaxWarning: invalid escape sequence '\m'
<>:23: SyntaxWarning: invalid escape sequence '\m'
<>:24: SyntaxWarning: invalid escape sequence '\m'
C:\Users\hoang\AppData\Local\Temp\ipykernel_11188\81933467.py:23: SyntaxWarning: invalid escape sequence '\m'
  axes[0].text(1.45, 0.285, "dashed = constant $\mu_Y$", fontsize=6.5, color=RED)
C:\Users\hoang\AppData\Local\Temp\ipykernel_11188\81933467.py:24: SyntaxWarning: invalid escape sequence '\m'
  axes[1].text(1.45, 0.585, "dashed = constant $\mu_Y$", fontsize=6.5, color=RED)


wrote fig_geometry.png


## In-sample fit vs held-out transfer, ridge vs LightGBM

Sources: `lgbm_metrics.json` (per-dimension $R^2$ on the shared 90k/10k token split),
`geometry_vi_en.json` (held-out sentences), and the n=1,000 generation table.

In [6]:
L = json.loads((ROOT / "research-phase01-ver02/results/lgbm_metrics.json").read_text(encoding="utf-8"))
r2r, r2l = np.array(L["r2_ridge_per_dim"]), np.array(L["r2_lgbm_per_dim"])
gen = json.loads((OUT / "rescored_n1000.json").read_text(encoding="utf-8"))

fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.6))
ax = axes[0]
ax.scatter(r2r, r2l, s=6, alpha=0.6, color=BLUE, edgecolors="none")
lim = [min(r2r.min(), r2l.min()), max(r2r.max(), r2l.max())]
ax.plot(lim, lim, ls="--", lw=1, color=GREY)
ax.set_xlabel("ridge $R^2$ per dimension")
ax.set_ylabel("LightGBM $R^2$ per dimension")
ax.set_title(f"in-sample: +{r2l.mean()-r2r.mean():.3f} $R^2$, 512/512 dims", fontsize=7.8)

ax = axes[1]
vals = [rows["map"]["retrieval@1"], rows["lgbm"]["retrieval@1"]]
ax.bar([0, 1], vals, color=[BLUE, ORANGE], width=0.55)
for i, v in enumerate(vals):
    ax.annotate(f"{v:.4f}", (i, v), ha="center", va="bottom", fontsize=7)
ax.axhline(rows["mean"]["retrieval@1"], ls=":", lw=1, color=RED)
ax.set_xticks([0, 1]); ax.set_xticklabels(["ridge", "LightGBM"])
ax.set_ylabel("retrieval@1, held-out sentences")
ax.set_title("held-out: ridge retrieves 1.6× more", fontsize=7.8)
ax.set_ylim(0, 0.105)

ax = axes[2]
vals = [gen["mapped"]["chrf"], gen["lgbm"]["chrf"]]
ax.bar([0, 1], vals, color=[BLUE, ORANGE], width=0.55)
for i, v in enumerate(vals):
    ax.annotate(f"{v:.1f}", (i, v), ha="center", va="bottom", fontsize=7)
ax.set_xticks([0, 1]); ax.set_xticklabels(["ridge", "LightGBM"])
ax.set_ylabel("chrF, held-out generation")
ax.set_title("text: the better fit reads worse", fontsize=7.8)
ax.set_ylim(0, 23)
save(fig, "fig_insample.png")


wrote fig_insample.png


## Generation: eight channels, one decoder, n = 1,000

Sources: `rescored_n1000.json` (first 1,000 pairs of the saved ver02 outputs, scored
with the original scorer) and `research-phase01-1.2/results/gen_variants_vi_en.json`.

In [7]:
V = {r["channel"]: r for r in json.loads(
    (ROOT / "research-phase01-1.2/results/gen_variants_vi_en.json").read_text(encoding="utf-8"))["rows"]}
G = {
    "native":          dict(bleu=gen["native"]["bleu"],   chrf=gen["native"]["chrf"],   nll=gen["native"]["nll"],   degen=gen["native"]["degenerate_rate"],   col=GREY),
    "ridge":           dict(bleu=gen["mapped"]["bleu"],   chrf=gen["mapped"]["chrf"],   nll=gen["mapped"]["nll"],   degen=gen["mapped"]["degenerate_rate"],   col=BLUE),
    "ridge+recolor":   dict(bleu=V["ridge+recolor.1"]["bleu"], chrf=V["ridge+recolor.1"]["chrf"], nll=V["ridge+recolor.1"]["nll"], degen=V["ridge+recolor.1"]["degenerate_rate"], col=GREEN),
    "LightGBM":        dict(bleu=gen["lgbm"]["bleu"],     chrf=gen["lgbm"]["chrf"],     nll=gen["lgbm"]["nll"],     degen=gen["lgbm"]["degenerate_rate"],     col=ORANGE),
    "procrustes":      dict(bleu=V["procrustes"]["bleu"], chrf=V["procrustes"]["chrf"], nll=V["procrustes"]["nll"], degen=V["procrustes"]["degenerate_rate"], col=PURPLE),
    "procr.+recolor":  dict(bleu=V["procrustes+.3"]["bleu"], chrf=V["procrustes+.3"]["chrf"], nll=V["procrustes+.3"]["nll"], degen=V["procrustes+.3"]["degenerate_rate"], col="#9e7bb5"),
    "identity":        dict(bleu=gen["identity"]["bleu"], chrf=gen["identity"]["chrf"], nll=gen["identity"]["nll"], degen=gen["identity"]["degenerate_rate"], col=LGREY),
    "copy readout":    dict(bleu=gen["copy"]["bleu"],     chrf=gen["copy"]["chrf"],     nll=gen["copy"]["nll"],     degen=gen["copy"]["degenerate_rate"],     col=RED),
}
order = list(G)
y = np.arange(len(order))[::-1]

fig, axes = plt.subplots(2, 2, figsize=(7.2, 4.2))
for ax, key, lab, xlim in zip(axes.ravel(),
                              ["bleu", "chrf", "nll", "degen"],
                              ["BLEU ↑", "chrF ↑", "reference NLL ↓", "degenerate rate ↓"],
                              [None, None, None, None]):
    vals = [G[c][key] for c in order]
    ax.barh(y, vals, color=[G[c]["col"] for c in order], height=0.68)
    for yi, v in zip(y, vals):
        ax.annotate(f"{v:.2f}" if key in ("bleu", "chrf") else f"{v:.3f}",
                    (v, yi), va="center", ha="left", fontsize=6.2,
                    xytext=(2, 0), textcoords="offset points")
    ax.set_yticks(y); ax.set_yticklabels(order, fontsize=6.5)
    ax.set_xlabel(lab)
    lo, hi = ax.get_xlim(); ax.set_xlim(0, hi * 1.18)
fig.suptitle("The decoder, not the metric, decides: BLEU prefers loops, chrF/NLL/degeneration do not",
             fontsize=8.5, y=1.01)
save(fig, "fig_generation.png")


wrote fig_generation.png


## Collapse, not strangeness

Sources: `research-phase01-1.2/results/anomaly_detection.json`,
`MAP_REPAIRS.md` (distance quantiles), and `per_sent_vi_en.npz` + `gen_vi_en_mapped.jsonl`
for the cosine/NLL join. The PCA panel is recomputed from
`research-phase01-1.2/data/anomaly_vectors.npz` with `M = X @ W + b`.


In [8]:
A = json.loads((ROOT / "research-phase01-1.2/results/anomaly_detection.json").read_text(encoding="utf-8"))
fr = A["flag_rates"]
det = [("isolation_forest", "Isolation Forest"), ("gmm", "GMM (8)"), ("autoencoder", "autoencoder")]

fig, axes = plt.subplots(2, 2, figsize=(7.2, 4.6))

ax = axes[0, 0]
x = np.arange(len(det)); w = 0.26
for i, (key, lab, col) in enumerate((("normal_holdout", "genuine (10 % hold-out)", GREY),
                                     ("mapped", "mapped", BLUE),
                                     ("identity_src", "identity", LGREY))):
    ax.bar(x + (i - 1) * w, [fr[d][key] for d, _ in det], w, label=lab, color=col)
ax.set_xticks(x); ax.set_xticklabels([n for _, n in det], fontsize=7)
ax.set_ylabel("flagged fraction")
ax.set_title("(a) one-class detectors: mapped = 0.000")
ax.legend(frameon=False, fontsize=6.3, loc="upper left")
ax.set_ylim(0, 1.12)
for xi in x:
    ax.annotate("0.000", (xi, 0.0), ha="center", va="bottom", fontsize=6.2, color=BLUE,
                xytext=(0, 2), textcoords="offset points")

ax = axes[0, 1]
zrows = [("genuine $Y$", 22.77, 16.65, 27.46, GREY),
         ("ridge", 6.31, 0.99, 9.85, BLUE),
         ("procrustes", 34.88, 16.07, 41.77, PURPLE),
         ("identity $X$", 37.89, 33.46, 45.90, LGREY),
         ("constant $\mu_Y$", 0.0, 0.0, 0.0, RED)]
for i, (lab, med, p05, p95, col) in enumerate(zrows):
    yy = len(zrows) - 1 - i
    ax.plot([p05, p95], [yy, yy], color=col, lw=3, solid_capstyle="round", alpha=0.75)
    ax.plot(med, yy, "o", color=col, ms=4)
    ax.annotate(f"{med:.1f}", (med, yy), xytext=(0, 5), textcoords="offset points",
                ha="center", fontsize=6.2, color=col)
ax.axvline(22.77, ls="--", lw=0.9, color=GREY)
ax.set_yticks(range(len(zrows))[::-1]); ax.set_yticklabels([r[0] for r in zrows], fontsize=6.8)
ax.set_xlabel("$\|z\|$ in the target frame (p05 – median – p95)")
ax.set_title("(b) the mapped cloud is 3.6× too tight")

ax = axes[1, 0]
z = np.load(ROOT / "research-phase01-1.2/data/anomaly_vectors.npz", mmap_mode="r")
X, Y = z["X"], z["Y"]
sel = np.arange(0, len(X), 30)
Xs, Ys = np.asarray(X[sel], np.float32), np.asarray(Y[sel], np.float32)
wmap = np.load(ROOT / "research-phase01-ver02/data/map_vi_en.npz")
Ms = Xs @ wmap["W"].astype(np.float32) + wmap["b"].astype(np.float32)
mu, sd = Ys.mean(0), Ys.std(0) + 1e-6
stack = np.concatenate([(Ys - mu) / sd, (Ms - mu) / sd, (Xs - mu) / sd])
from sklearn.decomposition import PCA
P = PCA(n_components=2, random_state=0).fit_transform(stack)
n = len(Ys)
for sl, lab, col, al, ss in ((slice(0, n), "genuine", GREY, 0.14, 2),
                             (slice(n, 2 * n), "mapped", BLUE, 0.55, 2.5),
                             (slice(2 * n, 3 * n), "identity", LGREY, 0.18, 2)):
    ax.scatter(P[sl, 0], P[sl, 1], s=ss, alpha=al, color=col, label=lab, edgecolors="none")
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
ax.set_title("(c) PCA of target-frame states")
ax.legend(frameon=False, fontsize=6.5, markerscale=3)
ax.grid(False)

ax = axes[1, 1]
ps = np.load(ROOT / "research-phase01-ver02/data/per_sent_vi_en.npz")
rows = [json.loads(l) for l in (ROOT / "research-phase01-ver02/results/gen_vi_en_mapped.jsonl").open(encoding="utf-8")]
cos = np.asarray(ps["map_sent_cos"])[:len(rows)]
nll = np.array([r["nll"] for r in rows])
ax.scatter(cos, nll, s=3, alpha=0.3, color=BLUE, edgecolors="none")
q = np.argsort(cos); k = max(len(cos) // 25, 1)
xs = [cos[q[j:j + k]].mean() for j in range(0, len(q), k)]
ys = [nll[q[j:j + k]].mean() for j in range(0, len(q), k)]
ax.plot(xs, ys, color=RED, lw=1.6)
rho = np.corrcoef(cos, nll)[0, 1]
ax.set_xlabel("per-sentence cosine (mapped vs true target)")
ax.set_ylabel("decoder NLL of the reference")
ax.set_title(f"(d) cosine does not predict NLL (r = {rho:+.3f})")
save(fig, "fig_collapse.png")


<>:27: SyntaxWarning: invalid escape sequence '\m'
<>:36: SyntaxWarning: invalid escape sequence '\|'
<>:27: SyntaxWarning: invalid escape sequence '\m'
<>:36: SyntaxWarning: invalid escape sequence '\|'
C:\Users\hoang\AppData\Local\Temp\ipykernel_11188\3413853807.py:27: SyntaxWarning: invalid escape sequence '\m'
  ("constant $\mu_Y$", 0.0, 0.0, 0.0, RED)]
C:\Users\hoang\AppData\Local\Temp\ipykernel_11188\3413853807.py:36: SyntaxWarning: invalid escape sequence '\|'
  ax.set_xlabel("$\|z\|$ in the target frame (p05 – median – p95)")


wrote fig_collapse.png


## The ranking flip

Left: retrieval@1 on the 4,000-vector bank (`MAP_REPAIRS.md`). Right: BLEU at n=1,000.
The orthogonal map wins the diagnostic and loses the task.

In [9]:
fig, ax = plt.subplots(figsize=(4.6, 3.0))
left = {"ridge": 0.0770, "ridge+recolor": 0.0915, "procrustes": 0.0938}
right = {"ridge": 2.717, "ridge+recolor": 3.171, "procrustes": 0.478}
col = {"ridge": BLUE, "ridge+recolor": GREEN, "procrustes": PURPLE}
offs = {"procrustes": 8, "ridge+recolor": 0, "ridge": -8}
for k in left:
    ax.plot([0, 1], [left[k], right[k] / 3.4], marker="o", ms=5, lw=1.8, color=col[k], label=k)
    ax.annotate(f"{left[k]:.4f}", (0, left[k]), xytext=(-7, offs[k]), textcoords="offset points",
                ha="right", va="center", fontsize=7, color=col[k])
    ax.annotate(f"{right[k]:.2f}", (1, right[k] / 3.4), xytext=(7, 0), textcoords="offset points",
                ha="left", va="center", fontsize=7, color=col[k])
ax.set_xlim(-0.25, 1.25); ax.set_xticks([0, 1])
ax.set_xticklabels(["retrieval@1\n(diagnostic)", "BLEU / 3.4\n(generation)"])
ax.set_ylabel("score (scaled to one axis)")
ax.set_title("The best map by retrieval is the worst by text")
ax.legend(frameon=False, fontsize=7, loc="center left")
save(fig, "fig_inversion.png")


wrote fig_inversion.png


## Recolor sweep and the generation result

Source: `research-phase01-1.2/results/recolor_sweep.json` and `gen_variants_vi_en.json`.

In [10]:
S = json.loads((ROOT / "research-phase01-1.2/results/recolor_sweep.json").read_text(encoding="utf-8"))
gen_t = S["genuine_zdist_med"]
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.5))
for ax, key, lab in ((axes[0], "ret_at_1", "retrieval@1"),
                     (axes[1], "zdist_med", "$\|z\|$ median")):
    for name, col in (("ridge", BLUE), ("procrustes", PURPLE)):
        pts = [(r["t"], r[key]) for r in S["sweep"] if r["map"] == name]
        ax.plot([p[0] for p in pts], [p[1] for p in pts], marker="o", ms=3, color=col, label=name)
    ax.set_xlabel("blend $t$ toward full recolor")
    ax.set_ylabel(lab)
    ax.legend(frameon=False, fontsize=6.5)
axes[1].axhline(gen_t, ls="--", lw=1, color=RED)
axes[1].annotate("genuine 22.8", (0.02, gen_t + 1), fontsize=6.3, color=RED)
axes[0].annotate("t = 0.1", (0.1, 0.0915), xytext=(0.12, 0.083), fontsize=6.3,
                 arrowprops=dict(arrowstyle="-", lw=0.6))

ax = axes[2]
names = ["ridge", "ridge+recolor.1", "procrustes", "procrustes+.3"]
lab = ["ridge", "+recolor\n(t=0.1)", "procrustes", "+recolor\n(t=0.3)"]
vals = [V[n]["bleu"] for n in names]
ax.bar(range(4), vals, color=[BLUE, GREEN, PURPLE, "#9e7bb5"], width=0.62)
for i, v in enumerate(vals):
    ax.annotate(f"{v:.2f}", (i, v), ha="center", va="bottom", fontsize=6.5)
ax.set_xticks(range(4)); ax.set_xticklabels(lab, fontsize=6.3)
ax.set_ylabel("BLEU")
ax.set_title("a 10 % correction buys +16 %")
save(fig, "fig_recolor.png")


<>:5: SyntaxWarning: invalid escape sequence '\|'
<>:5: SyntaxWarning: invalid escape sequence '\|'
C:\Users\hoang\AppData\Local\Temp\ipykernel_11188\1105002703.py:5: SyntaxWarning: invalid escape sequence '\|'
  (axes[1], "zdist_med", "$\|z\|$ median")):


wrote fig_recolor.png


Done. `figures/` now holds every panel referenced by `draft.md`.